# Representation-specific inverse and coordinate controls
Proof: [theory_main.md](theory_main.md). Ideal real-arithmetic inverse; actual Torch behavior is checked separately. No trained HSE advantage is asserted.

In [ ]:
import numpy as np
from experiments.p19.routing import weighted_headroom, conditional_regret
from experiments.p19.toy_routing import check_boundaries
from experiments.p19.affine_controls import fit_coordinates, ridge_fit, predict
check_boundaries()
np.testing.assert_allclose(weighted_headroom([[0,1],[1,0]],[.9,.1]),.1)

In [ ]:
rng=np.random.default_rng(27);d=2;s=5;q=8
rp=rng.normal(size=s);rt=rng.normal(size=q-s)
wp=np.eye(s)+.1*rng.normal(size=(s,s));wt=.1*rng.normal(size=(s,q-s));b=.1*rng.normal(size=s)
h=wp@rp+wt@rt+b
ii=np.tril_indices(d);diag=ii[0]==ii[1]
L=np.zeros((d,d));v=h[d:].copy();v[diag]=np.logaddexp(0,v[diag]);L[ii]=v
floor=.1;C=np.linalg.cholesky(L@L.T+floor*np.eye(d))
Li=np.linalg.cholesky(C@C.T-floor*np.eye(d));raw=Li[ii].copy();raw[diag]=np.log(np.expm1(raw[diag]))
hi=np.r_[h[:d],raw];recovered=np.linalg.solve(wp,hi-wt@rt-b)
np.testing.assert_allclose(recovered,rp,rtol=0,atol=1e-10)
print('Ideal complete-message inverse maximum error:',np.max(abs(recovered-rp)))
singular=wp.copy();singular[:,0]=0;delta=np.zeros(s);delta[0]=1
np.testing.assert_allclose(singular@rp,singular@(rp+delta),rtol=0,atol=1e-14)
print('Rank-deficient prefix permits an ambient-code collision')

## Same-head affine control
The control transmits (h,tail), before the statistical map. M and this control determine one another on the attainable image even when the raw head loses a direction of R. This does not claim a floating-point or learned-model guarantee.

In [ ]:
A=np.block([[wp,wt],[np.zeros((q-s,s)),np.eye(q-s)]])
offset=np.r_[b,np.zeros(q-s)];R=np.r_[rp,rt]
ha=A@R+offset
np.testing.assert_allclose(ha,np.r_[h,rt],rtol=0,atol=1e-14)
B=rng.normal(size=(3,q));intercept=rng.normal(size=3)
np.testing.assert_allclose(B@ha+intercept,(B@A)@R+B@offset+intercept,rtol=0,atol=1e-13)
np.testing.assert_allclose(hi,h,rtol=0,atol=1e-13)
hs=singular@rp+wt@rt+b
Ls=np.zeros((d,d));vs=hs[d:].copy();vs[diag]=np.logaddexp(0,vs[diag]);Ls[ii]=vs
Cs=np.linalg.cholesky(Ls@Ls.T+floor*np.eye(d))
Ls_rec=np.linalg.cholesky(Cs@Cs.T-floor*np.eye(d));vs_rec=Ls_rec[ii].copy();vs_rec[diag]=np.log(np.expm1(vs_rec[diag]))
np.testing.assert_allclose(np.r_[hs[:d],vs_rec],hs,rtol=0,atol=1e-12)
print('Same-head affine collapse and statistical inverse passed, including singular-prefix h recovery')

In [ ]:
rng=np.random.default_rng(7)
x=rng.normal(size=(64,4))*[.1,1,3,10];xt=rng.normal(size=(20,4))*[.1,1,3,10];y=rng.normal(size=(64,2))
center,coordinates,eigenvalues=fit_coordinates(x)
w,b=ridge_fit(x,y,center,coordinates['R'],.1);reference=predict(xt,center,coordinates['R'],w,b)
for name,a in coordinates.items():
    w,b=ridge_fit(x,y,center,a,.1,'matched');error=np.max(abs(predict(xt,center,a,w,b)-reference))
    assert error<1e-10
    print(name,'matched error',float(error))
w,b=ridge_fit(x,y,center,coordinates['whitened_R'],.1,'isotropic')
changed=np.max(abs(predict(xt,center,coordinates['whitened_R'],w,b)-reference))
assert changed>.01
print('Unmatched whitening changes the objective:',float(changed))

In [ ]:
source=np.array([[.1,.3],[.3,.1]]);target=source[:,::-1]
regret=conditional_regret(target,np.argmin(source,axis=1),[.5,.5])
np.testing.assert_allclose(regret,.2)
assert regret<=2*np.max(abs(target-source))+1e-12
print('Reversal sensitivity:',regret)

In [ ]:
r=np.array([-1.,0.,1.]);y=r*r
np.testing.assert_allclose(np.mean((y-y.mean())**2),2/9)
print('The nonlinear square witness is not evidence for an affine mean head.')